# Group Report - Predicting Following Day Rainfall in Australia


## 1.0 Introduction

#### 1.1 Background

Humans have been looking to the sky for thousands of years in hopes of forecasting the elements. Yet, only after the 1850s, did humanity begin systematically collecting meteorological data to scientifically predict weather conditions (Bell & Ogilvie, 1978). From the rain's immense impact on various industries, societal infrastructure, and general day-to-day life, predicting its occurrence is pivotal to many facets of society (Agbo, 2021). Particularly, The Australian Bureau of Meteorology has amassed over 10 years of weather data with consistent daily measurements of variables such as wind, humidity, temperature, and most importantly, rain.

#### 1.2 Central Question

Using the data collected in Australia and a classification approach, the research question of our project is:  
**"Will it rain tomorrow in Australia based on a set of meteorological characteristics from the previous day?"**

#### 1.3 Dataset

The dataset that we will use is the “Rain in Australia” (Young & Young, 2021). This dataset contains meteorological data across 10 years in Australia from 2007/10/31 to 2017/6/24 in various regions, collected by weather stations across Australia. The dataset contains daily minimum and maximum temperature, rainfall, strongest wind gust, evaporation, and sunshine, together with values at 9am and 3pm of temperature, humidity, wind, cloud and pressure. RainTomorrow is the target variable to predict. It means -- did it rain the next day, Yes or No? This column is Yes if the rain for that day was 1mm or more.

## 2.0 Methods & EDA


#### 2.1 Wrangling

To aid in our decision for predictor variables, we can visualize which columns are present with the most valid data (least NA columns). A larger sample of data would allow us to reduce the impact of factors such as random error in the observation process and improve the overall quality of the analysis. However, due to computational limits, we will need to address our sample size during our analysis.

In [ ]:
#load tidyverse and necessary packages!
library(tidyverse)
library(tidymodels)
library(modeldata)
install.packages("themis")
library(themis)
library(knitr)

In [ ]:
#load data into r and set the seed for the rest of the analysis. This will standardize our analysis for reproducibility.
set.seed(9784)

url <- "https://raw.githubusercontent.com/Andrewyx/DSCI100-Project-G16/main/weatherAUS.csv"
weather_data_raw <- read_csv(url)

In [ ]:
# Summarize each row by counting non-N/A cells. Renames the variables afterwards after conversion to a data frame
# This lets us see the variables containing the most amount of usable data, and thus choose our predictors!

no_na_data <- as.data.frame((colSums(!is.na(weather_data_raw)))) 
no_na_data <- cbind(rownames(no_na_data), no_na_data)
rownames(no_na_data) <- NULL
colnames(no_na_data) <- c("measurement","count")

count_tbl <- arrange(no_na_data, desc(count))
cat("Table 1: Summary of non-N/A Cells For Each Predictor.")
count_tbl

In [ ]:
#Remove categorical variables & variables with too many N/A values (Wrangle Data).
#Also remove redundant variables (MaxTemp & MinTemp) which increases the complexity of the model (Since we have Temp3pm & Temp9am).  

weather_data_clean <- weather_data_raw |>
                            select(-WindGustDir, -WindDir9am, -WindDir3pm, -RainToday, -Date, -Location) |>
                            select(-Sunshine, -Evaporation, -Cloud3pm, -Cloud9am) |>
                            select(-MaxTemp, -MinTemp) |>
                            transform(RainTomorrow = as.factor(RainTomorrow))

In [ ]:
# A sample of 8000 observations is drawn for our analysis. Split training and testing sets with a 75:25 training-testing split.
weather_data_reduced <- sample_n(weather_data_clean, 8000)
weather_split <- initial_split(weather_data_reduced, prop = 0.75, strata = RainTomorrow)
weather_train <- training(weather_split)
weather_test <- testing(weather_split)

#### 2.2 Summarizing Training Data
We now create two summary tables of the training data.

In [ ]:
glimpse(weather_train)

#### Means and Missing Rows
The first two summary tables present the means of the predictor variables and the number of rows with missing data.

In [ ]:
# Summary table of the mean of predictor variables and rows with missing data

summary_weather_train <- weather_train|>
  summarize(
    Mean_Rainfall = mean(Rainfall, na.rm = TRUE),
    Mean_WindGustSpeed = mean(WindGustSpeed, na.rm = TRUE),
    Mean_WindSpeed9am = mean(WindSpeed9am, na.rm = TRUE),
    Mean_WindSpeed3pm = mean(WindSpeed3pm, na.rm = TRUE),
    Mean_Humidity9am = mean(Humidity9am, na.rm = TRUE),
    Mean_Humidity3pm = mean(Humidity3pm, na.rm = TRUE),
    Mean_Pressure9am = mean(Pressure9am, na.rm = TRUE),
    Mean_Pressure3pm = mean(Pressure3pm, na.rm = TRUE),
    Mean_Temp9am = mean(Temp9am, na.rm = TRUE),
    Mean_Temp3pm = mean(Temp3pm, na.rm = TRUE))|>
    as.data.frame() |>
    pivot_longer(cols=everything(), names_to="Mean_Categories", values_to="Means")
cat("Table 2: Summary table of the mean of predictor variables.")
summary_weather_train


missing_data_table <- tibble(Rows_with_Missing_Data = sum(rowSums(is.na(weather_train)) > 0))
cat("Table 3: Number of rows with missing data.")
missing_data_table


#### Available Target Variable Data
The second summary table shows the counts of observations in our target RainTomorrow class. We observe here that the data appears to be heavily unbalanced.

In [ ]:
# Summary table for counts of RainTomorrow observation 

summary_num_obs<-weather_train|>
  group_by(RainTomorrow)|>
  summarize(count=n())

cat("Table 4: Summary table for counts of RainTomorrow observation.")
summary_num_obs

#### 2.3 Visualizing training data

Now we compare the distributions of each of the predictor variables.

In [ ]:
# Visualize distributions of variables via histogram

options(repr.plot.height = 13, repr.plot.width = 7)
library(ggplot2)
library(gridExtra)

weather_predictors <- na.omit(select_if(weather_train, is.numeric))
plots<-lapply(names(weather_predictors), function(col)  {
    ggplot(weather_predictors, aes(x = .data[[col]])) +
      geom_histogram(fill = "skyblue", color = "black",bins=20) +
      labs(title = paste("Histogram of", col),
           x = col, y = "Frequency" ) +
      theme_minimal()
  })
grid.arrange(grobs = plots, ncol = 2, top = "Figure 1: Distribution of 10 predictor variables.")



## 3.0 Data Analysis & Results

First, we must remove the previously found NA rows from our training and testing data.

In [ ]:
# Omit the NA rows from the training and testing data
weather_train_clean <- weather_train |>
                            na.omit()

weather_test_clean <- weather_test |>
                            na.omit()

#### 3.1 Tuning the Model for K

Since we are using knn classification to conduct our data analysis, we must find a suitable K for our model. Through the EDA we observed that the data was heavily unbalanced, hence we will use downsampling to balance the data. Downsampling is chosen in this case due to the large dataset and the computational limitations of this project. We will also need to scale and center our data to ensure consistency across Euclidean distances.

In [ ]:
# Finding K with tuning and including downsampling in our recipe. We make our recipe with all predictors of our weather_train_clean data set.
knn_spec_tune <- nearest_neighbor(weight_func = "rectangular", neighbors = tune()) |>
                         set_engine("kknn") |>
                         set_mode("classification")

rain_recipe <- recipe(RainTomorrow ~ ., data=weather_train_clean)|>
                        step_downsample(under_ratio = 1, RainTomorrow) |>
                        step_scale(all_predictors()) |>
                        step_center(all_predictors())
rain_recipe

#### 3.2 Cross Validation

We can now test a range of K's with cross-validation to find the best-suited K for our model. For this, we have chosen a range of K's from 1 to 70  incrementing by K's of 5. K values beyond the range of 70 would be too computationally costly to justify the diminishing returns to accuracy (See graph of K).

In [ ]:
# Cross-validation and finding K for tests of values from 1 to 70
k_vals <- tibble(neighbors = seq(from = 1, to = 70, by = 5))
rain_vfold <- vfold_cv(weather_train_clean, v = 5, strata = RainTomorrow)

# Create a workflow and collect the metrics from our validations.
knn_results <- workflow() |>
  add_recipe(rain_recipe) |>
  add_model(knn_spec_tune) |>
  tune_grid(resamples = rain_vfold, grid = k_vals)|>
  collect_metrics()

#### Accuracies
Analyzing our knn results through their accuracies, we can plot each corresponding mean accuracy vs K to obtain a graph and visually inspect the effect of K on accuracy for our model. 

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5)

# Store all mean accuracy measurements and plot against their corresponding K values.
accuracies <- knn_results |>
  filter(.metric == "accuracy")

accuracy_vs_k <- ggplot(accuracies, aes(x = neighbors, y = mean)) +
  geom_point() +
  geom_line() +
  labs(title="Figure 2: Plot of estimated accuracy versus the number of neighbors.", x= "Neighbors", y = "Accuracy Estimate") +
  theme(text = element_text(size = 12))
accuracy_vs_k

#### 3.3 Best K Value
Our best K value can now be pulled from our accuracy data frame. Since we see from the above graph that accuracy plateaus with increasing K, this implies diminishing returns for increasing K. For us, the best K within our range appears to be 56. From this, we observe that our model for k=56 appears to have an accuracy of 79.1%.

In [16]:
# Pull the k based on highest mean accuracy as our desired K value
cat("Table 5: K with highest mean accuracy.")
accuracies |>
    arrange(desc(mean)) |>
    head(1)

best_k <- accuracies |>
        arrange(desc(mean)) |>
        head(1) |>
        pull(neighbors)

Table 5: K with highest mean accuracy.

neighbors,.metric,.estimator,mean,n,std_err,.config
<dbl>,<chr>,<chr>,<dbl>,<int>,<dbl>,<chr>
56,accuracy,binary,0.7906092,5,0.007615814,Preprocessor1_Model12


#### 3.4 Final KNN Model
We can now build our classification model with the ideal K from above, fitting it with our clean training data.

In [ ]:
# Build the final knn model with our chosen k value from the table above.
knn_spec_final <- nearest_neighbor(weight_func = "rectangular", neighbors = best_k) |>
     set_engine("kknn") |>
     set_mode("classification")

knn_fit <- workflow() |>
  add_recipe(rain_recipe) |>
  add_model(knn_spec_final) |>
  fit(data=weather_train_clean)
knn_fit

#### 3.5 Results
Predicting RainTomorrow on our testing data nets us a final accuracy, precision, and recall value. Our final accuracy lies around 80.1%. These results are shown below alongside precision (52.7%) and recall (75.6%).

In [ ]:
# Testing and Evaluating the model

rain_test_predictions <- predict(knn_fit, weather_test_clean) |>
  bind_cols(weather_test_clean) 

cat("Table 6: Accuracy of the model.")
rain_test_predictions |>
  metrics(truth = RainTomorrow, estimate = .pred_class) |>
  filter(.metric == "accuracy")

cat("Table 7: Precision of the model.")
rain_test_predictions |>
    precision(truth = RainTomorrow, estimate = .pred_class, event_level="second")

cat("Table 6: Recall of the model.")
rain_test_predictions |>
    recall(truth = RainTomorrow, estimate = .pred_class, event_level="second")

#### 3.6 Analysis Visualization
To better visualize our results, we can use a confusion matrix in the form of a heatmap to compare our Truth and Prediction values.

In [ ]:
# Visualize with Confusion Matrix
confusion <- rain_test_predictions |>
             conf_mat(truth = RainTomorrow, estimate = .pred_class)
confusion_plot<-autoplot(confusion, type = "heatmap")+
   ggtitle("Figure 3:Confusion Matrix Heatmap.")
confusion_plot

## 4.0 Discussion

#### 4.1 Outcomes and Expectations

Using the K-nearest neighbours algorithm, we've developed a classifier to predict whether there will be rain the next day in Australia, based on a specific set of weather characteristics as input. This model achieves an accuracy of 80.1%, indicating the proportion of overall correct predictions. Additionally, it gives a precision of 52.7%, highlighting the proportion of correctly predicted rainy days among all predicted rainy days. Lastly, the recall is 75.6%, indicating the proportion of correctly predicted rainy days among all actual rainy days. These metrics provide a comprehensive evaluation of our classifier's performance. 

In terms of our expectations, we aimed for our classification model to be able to predict results based on learning correlations within the data and without simply memorizing data points. Achieving an 80.1% accuracy aligns fairly well with our anticipated outcomes. This accuracy rate suggests that our model's predictions closely align with the actual labels assigned to the observations in the test set. Our recall also falls within a rather expected range of 75.6%, which was rather expected from our perspective. In contrast, it was unexpected for our model to have such low precision after our data analysis despite our additional efforts to reduce noise in the data by removing redundant predictors and through tuning our model. 

#### 4.2 Limitations


The classifier is likely unacceptable with a precision of 52.7%. However, precision quantifies how many of the positive predictions the classifier made were true positives. In this context, it indicates the classifier's tendency to produce false positives and predict rain when in reality none occurs. 

From the application standpoint of predicting rain, a higher recall is more desirable from an urban perspective as it would be more important for citizens to know for certain that a given day will not rain. This is likewise reflected in our high recall results. It is far more important to reduce the number of false negatives (reduce instances of predicting no rain, but rain occurs) as unexpected incremental weather can result in severe economic ramifications, unforeseen accidents, or disruptions (Leal et al., 2019). Hence, while our precision is indeed lower than expected, our higher recall seems to be much more important as a metric to gauge the effectiveness of our classifier.

#### 4.3 Significance

The significance of this analysis lies in the immense impact that weather and in particular, rain, has on society. Being able to predict rain is not only beneficial for day-to-day life but quintessential for industries such as agriculture, tourism, and urban development (Agbo, 2021). Furthermore, due to factors such as global warming and climate change leading to the increasing frequency of extreme weather, the necessity of preparing for such events is growing evermore paramount; the consequences of unexpected rainfall depending on its severity can result in significant material damage from a societal perspective (Leal et al., 2019).  

#### 4.4 Extended/Further Questions

After conducting this investigation, we had several new questions that surfaced from the results of our analysis. Specifically, concerning our rather low precision, how could we improve our analysis to produce a higher precision rating without sacrificing recall? This follows suit with our second question, which is whether other data analysis/classification techniques outside of KNN could lead to better results. Finally, looking into the specific results of our case, a question this investigation could lead to is how climate change has affected the rain patterns in Australia, and in turn, affected the results of our classification model.

## 5.0 References

Agbo, E. P. (2021). The role of statistical methods and tools for weather forecasting and modeling. In IntechOpen eBooks. https://doi.org/10.5772/intechopen.96854 

Bell, W., & Ogilvie, A. E. J. (1978). Weather compilations as a source of data for the reconstruction of european climate during the medieval period. Climatic Change, 1(4). https://doi.org/10.1007/bf00135154 

Leal, M., Boavida-Portugal, I., Fragoso, M., & Ramos, C. (2019). How much does an extreme rainfall event cost? Material damage and relationships between insurance, rainfall, land cover and urban flooding. Hydrological Sciences Journal, 64(6), 673–689. https://doi.org/10.1080/02626667.2019.1595625

Pfister, C., White, S., & Mauelshagen, F. (2018). General Introduction: Weather, Climate, and Human History. In Palgrave Macmillan UK eBooks (pp. 1–17). https://doi.org/10.1057/978-1-137-43020-5_1

Young, J., & Young, A. (2021). Rain in Australia. Kaggle: Your Machine Learning and Data Science Community. Retrieved April 12, 2024, from https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package
